# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arslaniqbalwah/flyrank-ml-internship-arslan/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
The unit of analysis is a single piece of content (represented by a pseudonymized content_id). The time window is a 90-day historical snapshot. One row = one unique page's aggregated 90-day search and traffic performance.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

total_rows = len(df)
unique_ids = df['content_id'].nunique()

print(f"Total rows: {total_rows}")
print(f"Unique content IDs: {unique_ids}")
print(f"Does 1 row = 1 content item? {'Yes' if total_rows == unique_ids else 'No'}")

Total rows: 30000
Unique content IDs: 30000
Does 1 row = 1 content item? Yes


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: impressions_90d, sessions_90d, content_age_days (The behavioral signals the model learns from).

Label: trend_direction (Our proxy target to define if a page is decaying).

Context: content_id, client_id (Needed to trace recommendations back to the client, but never fed to the model).

Excluded: Any raw URL strings or client names (already anonymized in this dataset) because we want the model to learn structural patterns, not memorize specific clients.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = ['impressions_90d', 'sessions_90d', 'content_age_days']
label = 'trend_direction'
context = ['content_id', 'client_id']

print("Data Contract Mapping:")
print(f"Features: {features}")
print(f"Label: {label}")
print(f"Context: {context}")

Data Contract Mapping:
Features: ['impressions_90d', 'sessions_90d', 'content_age_days']
Label: trend_direction
Context: ['content_id', 'client_id']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I am verifying the contract by checking for duplicate grains (ensuring content_id is truly unique) and auditing our selected features and label for missing data that could break the pipeline.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

duplicate_grain = df.duplicated(subset=['content_id']).sum()
missing_values = df[features + [label] + context].isnull().sum()

print(f"Duplicate grains (content_id): {duplicate_grain} (Should be 0)")
print("\nMissing values check:")
print(missing_values)

Duplicate grains (content_id): 0 (Should be 0)

Missing values check:
impressions_90d     0
sessions_90d        0
content_age_days    0
trend_direction     0
content_id          0
client_id           0
dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data only shows observed search metrics within a 90-day window. It can never tell us why a page is declining (e.g., if a competitor published better content, or if a Google Core Update hit). Furthermore, the 90-day window means we cannot see full-year seasonality (e.g., a Christmas page naturally dropping in January).

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Known Data Limits:")
print("1. Lacks external context (competitor actions, SERP layout changes).")
print("2. 90-day window misses macro seasonal trends.")
print("3. Purely observational; correlation does not guarantee causation.")

Known Data Limits:
1. Lacks external context (competitor actions, SERP layout changes).
2. 90-day window misses macro seasonal trends.
3. Purely observational; correlation does not guarantee causation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.